In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sn
import yfinance as yf

In [3]:
ticker = '^NSEI'
start_date = '2014-01-01'
end_date = '2025-01-01'

df = yf.download(ticker,start_date,end_date)
df

/tmp/ipykernel_4526/1502302891.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker,start_date,end_date)
[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,^NSEI,^NSEI,^NSEI,^NSEI,^NSEI
Date,,,,,
2014-01-02,6221.149902,6358.299805,6211.299805,6301.250000,158100
2014-01-03,6211.149902,6221.700195,6171.250000,6194.549805,139000
2014-01-06,6191.450195,6224.700195,6170.250000,6220.850098,118300
2014-01-07,6162.250000,6221.500000,6144.750000,6203.899902,138600
2014-01-08,6174.600098,6192.100098,6160.350098,6178.049805,146900
...,...,...,...,...,...
2024-12-24,23727.650391,23867.650391,23685.150391,23769.099609,177700
2024-12-26,23750.199219,23854.500000,23653.599609,23775.800781,177700


**A1 - STATIONARITY**

In [4]:
from statsmodels.tsa.stattools import adfuller

In [5]:
result = adfuller(df['Close']['^NSEI'])
print(result[1])

0.97864667584301


In [6]:
daily_returns_matrix = df['Close']['^NSEI'].pct_change().dropna()
daily_returns_matrix.dropna(inplace=True)
result = adfuller(daily_returns_matrix)
print(result[1])

7.245295456352895e-27


**A2 - LOOK AHEAD BIAS**

In [7]:
from sklearn.model_selection import train_test_split

In [8]:
returns = df['Close']['^NSEI'].pct_change().dropna()
returns = returns.reset_index(drop=True)


df_returns = pd.DataFrame()
df_returns['returns'] = returns.values
df_returns['lag1'] = df_returns['returns'].shift(1)
df_returns['target'] = (df_returns['returns'] > 0).astype(int)
df_returns.dropna(inplace=True)

df_returns

,returns,lag1,target
1,-0.003172,-0.001607,0
2,-0.004716,-0.003172,0
3,0.002004,-0.004716,1
4,-0.001012,0.002004,0
5,0.000503,-0.001012,1
...,...,...,...
2693,-0.001086,0.007035,0
2694,0.000950,-0.001086,1
2695,0.002661,0.000950,1
2696,-0.007076,0.002661,0


**A3 - WALK-FORWARD VALIDATION**

In [9]:
from sklearn.linear_model import LogisticRegression

In [10]:
X = df_returns[['lag1']].values
y = df_returns['target'].values

# Split 1: shuffled (wrong way)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True)
model = LogisticRegression()
model.fit(X_train, y_train)
print("Shuffled accuracy:", model.score(X_test, y_test))

# Split 2: time-ordered (correct way)
split = int(len(X) * 0.8)
X_train2, X_test2 = X[:split], X[split:]
y_train2, y_test2 = y[:split], y[split:]
model2 = LogisticRegression()
model2.fit(X_train2, y_train2)
print("Time-ordered accuracy:", model2.score(X_test2, y_test2))


Shuffled accuracy: 0.5166666666666667
Time-ordered accuracy: 0.562962962962963


In [11]:
accuracies = []
for i in range(1500, len(X), 50):
  X_train = X[:i]
  y_train = y[:i]
  X_test = X[i:i+50]
  y_test = y[i:i+50]
  model = LogisticRegression()
  model.fit(X_train, y_train)
  model.predict(X_test)
  accuracies.append(model.score(X_test, y_test))

print(np.mean(accuracies))

0.557677304964539


**A4 - OVERFITTING IN FINANCE**

In [12]:
from sklearn.tree import DecisionTreeClassifier

X = df_returns[['lag1']]
y = df_returns['target']
X_train = X.iloc[0:1501]
y_train = y.iloc[0:1501]

model = DecisionTreeClassifier()
model.fit(X_train, y_train)
print("Training accuracy:",model.score(X_train, y_train))
print("Test accuracy:",model.score(X.iloc[1501:], y.iloc[1501:]))

Training accuracy: 1.0
Test accuracy: 0.5376254180602007


In [13]:
model = DecisionTreeClassifier(max_depth=3)
model.fit(X_train, y_train)
print("Training accuracy:",model.score(X_train, y_train))
print("Test accuracy:",model.score(X.iloc[1501:], y.iloc[1501:]))

Training accuracy: 0.5489673550966022
Test accuracy: 0.5451505016722408


**FEATURE ENGINEERING**

In [14]:
df_returns['lag5'] = df_returns['returns'].shift(5)
df_returns.dropna(inplace=True)
df_returns['rolling mean'] = df_returns['returns'].rolling(window=10).mean().shift(1)
df_returns['rolling volatility'] = df_returns['returns'].rolling(window=10).std().shift(1)
df_returns.dropna(inplace=True)
df_returns.head(15)

,returns,lag1,target,lag5,rolling mean,rolling volatility
16,-0.020888,-0.012434,0,0.006755,0.001570,0.009014
17,-0.001565,-0.020888,0,0.001562,-0.002160,0.009867
18,-0.000979,-0.001565,0,0.003983,-0.001824,0.009819
19,-0.007606,-0.000979,0,0.001057,-0.003188,0.008432
20,0.002601,-0.007606,1,-0.012434,-0.003917,0.008471
21,-0.014402,0.002601,0,-0.020888,-0.002751,0.008487
22,-0.000150,-0.014402,0,-0.001565,-0.004867,0.008491
23,0.003583,-0.000150,1,-0.000979,-0.005038,0.008363
24,0.002308,0.003583,1,-0.007606,-0.005078,0.008316
25,0.004456,0.002308,1,0.002601,-0.004953,0.008427


In [20]:
volume = df['Volume']['^NSEI'].reset_index(drop=True)
df_returns['rolling volume 5'] = volume.rolling(5).mean().shift(1)
df_returns['rolling volume 10'] = volume.rolling(10).mean().shift(1)
df_returns['volume ratio'] = (volume/volume.rolling(10).mean()).shift(1)
df_returns = df_returns.reindex(df_returns.index)
print(df_returns.head(10))

     returns      lag1  target      lag5  rolling mean  rolling volatility  \
16 -0.020888 -0.012434       0  0.006755      0.001570            0.009014   
17 -0.001565 -0.020888       0  0.001562     -0.002160            0.009867   
18 -0.000979 -0.001565       0  0.003983     -0.001824            0.009819   
19 -0.007606 -0.000979       0  0.001057     -0.003188            0.008432   
20  0.002601 -0.007606       1 -0.012434     -0.003917            0.008471   
21 -0.014402  0.002601       0 -0.020888     -0.002751            0.008487   
22 -0.000150 -0.014402       0 -0.001565     -0.004867            0.008491   
23  0.003583 -0.000150       1 -0.000979     -0.005038            0.008363   
24  0.002308  0.003583       1 -0.007606     -0.005078            0.008316   
25  0.004456  0.002308       1  0.002601     -0.004953            0.008427   

    rolling volume 5  rolling volume 10  volume ratio  
16          137820.0           139350.0      0.861859  
17          136340.0         